# Spoken Language Processing for Customer Insights

**Basis:** Spoken Language Processing in Python (Chapters 1-4) | datacamp  

---

## Notebook Rules

- Worked with a simulated customer support call*

In [1]:
!pip install SpeechRecognition
!pip install pydub
import wave
import numpy as np
import speech_recognition as sr
from pydub import AudioSegment

print("Libraries loaded.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 47.4 MB/s eta 0:00:00
Libraries loaded. Note: You must provide a valid .wav file named 'support_call.wav' to run the code cells.


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


## 1. Audio Data Inspection


In [2]:
#for 1 call_1_stereo_formatted.wav
support_call_1 = wave.open("call_1_stereo_formatted.wav", "r")

frame_rate = support_call_1.getframerate()
channels = support_call_1.getnchannels()
frames = support_call_1.getnframes()

duration = frames / frame_rate
print("Call 1")
print("Frame rate:", frame_rate)
print("Channels:", channels)
print("Duration:", duration, "seconds")

Call 1
Frame rate: 32000
Channels: 2
Duration: 54.888 seconds


In [3]:
#for 2 call_2_stereo_native.wav
support_call_2 = wave.open("call_2_stereo_native.wav", "r")

frame_rate = support_call_2.getframerate()
channels = support_call_2.getnchannels()
frames = support_call_2.getnframes()

duration = frames / frame_rate
print("Call 2")
print("Frame rate:", frame_rate)
print("Channels:", channels)
print("Duration:", duration, "seconds")

Call 2
Frame rate: 32000
Channels: 1
Duration: 52.756 seconds



Frame rate in this case is important because it shows the audio quality and how many sound samples are recorded per second. If the frame rate is too low or not supported by a transcription API, the speech may be recognized incorrectly. For business, this can lead to wrong customer call transcripts, wrong sentiment analysis, and poor customer insights

# 2. Basic Transcription


In [32]:
recognizer = sr.Recognizer()
wav_file = sr.AudioFile("call_1_stereo_formatted.wav")

with wav_file as source:
    call_1_audio = recognizer.record(source)

call_text = recognizer.recognize_google(call_1_audio)

print("Call transcription:")
print(call_text)

Call transcription:
hello welcome to Acme Studio support line my name is Daniel how can I best help you hey Danielle this is John I've recently bought a smartphone from you guys 3 weeks ago and I'm already having issues with it oh no that's not good to hear John let's let's get your serial number and then we can we can set up a way to fix it for you one second let me grab my cereal number it is 417-7577 I'm very displeased how long do you reckon this is going to take me on hold for about an hour now John we're going to try out best thing I'm just I'm just really really really really really really really just I've been trying to contact support for the past past 3:45


Converting voice to text makes sentiment analysis scalable because text is much easier for software to process than raw audio.
When customer calls are transcribed, the company can automatically analyze thousands of conversations instead of listening to each call manually. The text can be used to detect positive, negative, or neutral emotions, find repeated complaints, identify angry customers, and track common service issues. Voice-to-text turns unstructured audio into searchable and analyzable text data, which allows sentiment analysis to be done quickly, consistently, and at large scale

## 3. Audio Manipulation with PyDub

In [33]:
support_call = AudioSegment.from_file("call_1_stereo_formatted.wav")

support_call = support_call.set_channels(1)
support_call = support_call.set_frame_rate(16000)

support_call.export("standardized_call.wav", format="wav")

print("Channels:", support_call.channels)
print("Frame rate:", support_call.frame_rate)

Channels: 1
Frame rate: 16000


Many transcription models require mono audio and businesses can face a risk if they are ignore kinda requirements. Because, ignoring technical requirements can reduce transcription accuracy. For business, this means customer calls may be converted into wrong text, leading to incorrect sentiment analysis, missed complaints, wrong reports, and poor decisions about customer service

## 4. Helper Functions for Pipeline Automation


In [38]:
def transcribe_and_stats(file_path):
    audio = AudioSegment.from_file(file_path)
    print("Duration:", len(audio) / 1000, "seconds")

    recognizer = sr.Recognizer()

    audio_file = sr.AudioFile(file_path)
    with audio_file as source:
        audio_data = recognizer.record(source)

    text = recognizer.recognize_google(audio_data)
    print("Transcribed text:")
    print(text)

In [39]:
transcribe_and_stats("standardized_call.wav")

Duration: 54.888 seconds
Transcribed text:
hello welcome to Acme Studio support line my name is Daniel how can I best help you hey Danielle this is John I've recently bought a smartphone from you guys three weeks ago and I'm already having issues with it oh no that's not good to hear John let's let's get your serial number and then we can we can set up a way to fix it for you one second let me grab my cereal number it is 417-7577 I'm very displeased how long do you reckon this is going to take me on hold for about an hour now John we're going to try out best thing we'll get this support case we're on I'm just I'm just really really really really really really really just I've been trying to contact support for the past past 3:45


## 5. Reflection

Transcribing customer calls can create privacy risks because calls may include personal information such as names, phone numbers, addresses, or account details. Companies should inform customers that calls may be recorded and transcribed. They should also protect transcripts, limit access to them and avoid using customer data without permission
